In [ ]:
# read request list
import os
import pandas as pd

base_path = 'd:\\tapestry\\app_chn_competitor_report/flask'
app_static = base_path + '\\app\\static'
data_path = os.path.join(app_static, "data")
list = pd.read_excel(os.path.join(data_path, "list.xlsx"),sheet_name='data')
# ly_update_enable_list = pd.read_excel(os.path.join(data_path, "list.xlsx"),sheet_name='ly_update_enable')

list = list.where(pd.notna(list), None)

# calculate manager code and brand code
code_converter = lambda x: x.lower().strip().replace(" ", "_")
list['manager_code'] = list['manager'].map(code_converter)
list['brand_code'] = list['brand'].map(code_converter)

list

In [ ]:
# list.to_dict(orient='records')

In [ ]:
managers_list_df = list.groupby('manager_code').agg(
    manager=('manager', 'first'),  # Get the first manager for each manager_code
    store_count=('store', 'nunique')  # Count distinct stores
).reset_index()

managers_list_df.head()

In [ ]:
managers = managers_list_df.drop_duplicates().to_dict(orient='records')
managers

In [ ]:
manager_code = 'gigi_zou'
stores = list[list['manager_code']==manager_code][['store', 'store_name', 'status']]
stores = stores.drop_duplicates().to_dict(orient='records')
stores

In [ ]:
the_year = 2024 
the_month = 9
store_code = 'CNOCF10'

list_ly = list.copy(deep=True)
list_ly['year'] = int(the_year) - 1

df = pd.concat([list, list_ly])


store_data = df[df['store']==store_code]
store_data.head()

In [ ]:
def extend_df(list_df, store_code):
    list_ly = list.copy(deep=True)
    list_ly['year'] = int(the_year) - 1
    df = pd.concat([list, list_ly])
    df = df[df['store']==store_code]
    return df

extend_df(list, store_code)

In [ ]:
# query from database, get data, and create a data frame 
sales_data = pd.read_csv(os.path.join(data_path, "COMPETITOR_SALES_COLLECTION.csv"))
sales_data.columns = ["year", "month", "manager_db", "store", "store_name_db", "brand", "sales", "comp", "comment","store_category_db"]
sales_data = sales_data.astype({'year': 'int64', 'month': 'int64', 'manager_db': 'object', 'store': 'object',
                                'store_name_db': 'object', 'brand': 'object', 'sales': 'float64', 'comp': 'object', 'comment': 'object',
                                })


# merge database data and list data
merged_data = store_data.merge(sales_data
                        , on=["year", "month", "store", "brand"]
                        , how="left")


merged_data = merged_data[["year", "month", "manager", "store", "store_name", "status", "brand", "sales", "comp", "comment", "store_category", "brand_code"]]

merged_data.head()
# merged_data.shape, store_data.shape, sales_data.shape

In [ ]:
sales_data.to_dict(orient='records')

In [ ]:
def rotate_df(merged_data):
    # divide data into two years
    ty_database = merged_data[(merged_data['year'] == the_year)]
    ly_database = merged_data[(merged_data['year'] == the_year-1)]

    # rename sales to ly_sales for ly data
    ly_database.rename(columns={"sales": "ly_sales"}, inplace=True)

    # exclude comment because only ty comments would be displayed and updated
    ly_database = ly_database[["year", "month", "manager", "store", "store_name", "brand", "ly_sales"]]

    ly_database["year"] = ly_database["year"] + 1
    # merge ly sales into ty data for display
    combined_sales = ty_database.merge(ly_database, on=["year", "month", "manager", "store", "store_name", "brand"], how="inner")

    return combined_sales




In [ ]:
# divide data into two years
ty_database = merged_data[(merged_data['year'] == the_year)]
ly_database = merged_data[(merged_data['year'] == the_year-1)]

# rename sales to ly_sales for ly data
ly_database.rename(columns={"sales": "ly_sales"}, inplace=True)

# exclude comment because only ty comments would be displayed and updated
ly_database = ly_database[["year", "month", "manager", "store", "store_name", "brand", "ly_sales"]]

ly_database["year"] = ly_database["year"] + 1
# merge ly sales into ty data for display
combined_sales = ty_database.merge(ly_database, on=["year", "month", "manager", "store", "store_name", "brand"], how="inner")


brands = list[["brand", "brand_code"]].drop_duplicates().to_dict(orient="records")
brands = {m["brand"]: m["brand_code"] for m in brands}
combined_sales["brand_code"] = combined_sales['brand'].map(brands)


# add comp column
# when ty_sales is null or ly_sales is null, comp = 'N', else comp = 'Y'
combined_sales[['sales','ly_sales']].fillna(0, inplace=True)
combined_sales["comp"] = combined_sales.apply(lambda x: 'Y' if x["sales"] > 0 and x["ly_sales"] > 0 else 'N', axis=1)
combined_sales.sort_values(by=["ly_sales","sales"], ascending = False, inplace=True)

# reformat to list of dict and send the data to data collection page
# records = combined_sales.to_dict(orient="records")


combined_sales.head()

In [ ]:
combined_sales.loc[combined_sales['sales']==2101, 'comment'] = 'hi there data changed'

In [ ]:


# fetch last year sales data from database
sales_data = pd.read_csv(os.path.join(data_path, "COMPETITOR_SALES_COLLECTION.csv"))
sales_data.columns = sales_data.columns.str.lower()
sales_data = sales_data.astype({'year': 'int64', 'month': 'int64', 'manager_name': 'object', 'store_code': 'object',
                                'store_name': 'object', 'brand_name': 'object', 'amount': 'float64', 'comp': 'object', 'comment': 'object',
                                })
ly_db_sales_data = sales_data[(sales_data['year']==the_year) & (sales_data['month']==the_month) & (sales_data['store_code']==store_code)]
ly_db_sales_data = ly_db_sales_data[['store_code','brand_name','comp','comment']]


combined_sales = combined_sales.rename(columns={
    'sales': 'amount', 
    'manager': 'manager_name',
    'store': 'store_code',
    'brand': 'brand_name',
    })

ly_combined_sales = combined_sales.copy(deep=True)
ly_combined_sales['amount'] = ly_combined_sales['ly_sales']
ly_combined_sales['year'] = ly_combined_sales['year'] - 1
ly_combined_sales = ly_combined_sales[["year", "month", "manager_name", "store_code", "store_name", "brand_name", "amount", "store_category"]]


ly_combined_sales = ly_combined_sales.merge(ly_db_sales_data,on=['store_code','brand_name'],how='left')

col_list = ["year", "month", "manager_name", "store_code", "store_name", "brand_name", "amount", "comp", "comment", "store_category"]

combined_sales = combined_sales[col_list]
ly_combined_sales = ly_combined_sales[col_list]

df_form_final = pd.concat([combined_sales,ly_combined_sales]).reset_index(drop=True)
df_form_final['year'] = df_form_final['year'].astype('int')
df_form_final['month'] = df_form_final['month'].astype('int')
df_form_final.loc[df_form_final['amount'] == '','amount'] = 0
df_form_final['amount'] = df_form_final['amount'].astype('double')


df_form_final